<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/step4_image_feature_extract/image_feature_reorder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Đọc file đã được extract image feature *.b sắp sếp lại cho đúng thứ tự với df

Cần file df_meta.preprocessed.parquet của step3 phải được tạo ra trước đó

## Bước này tạo files:

- image_feat.npy (đọc file image_features.*.b tìm có text mà không có hình fill giá trị mặc định)

In [ ]:
import os
import array
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014/step3_text_feature_extract"

In [ ]:
!rm -rf data

In [ ]:
!mkdir -p "./data/baby/2014"

In [ ]:
!cp -r "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014/step3_text_feature_extract" "./data/baby/2014/step3_text_feature_extract"

In [ ]:
!cp -r "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014/step4_image_feature_extract" "./data/baby/2014/step4_image_feature_extract"

In [ ]:
PATH = "./data/baby/2014"

In [ ]:
df = pd.read_parquet(os.path.join(PATH, "step3_text_feature_extract", "df_meta.preprocessed.parquet"))

In [ ]:
df_meta = df[["itemID", "asin", "title"]].copy()
df_meta.shape

In [ ]:
df_meta = df[["itemID", "asin", "title"]].copy()
df_meta.shape

In [ ]:
df_meta.info()

In [ ]:
df_meta[:5]

In [ ]:
# df_filtered_image = pd.read_parquet(os.path.join(PATH, "step4_image_feature_extract", "df_filtered_image.parquet"))
# df_filtered_image.drop(columns=["title"], inplace=True)
# df_filtered_image.shape

In [ ]:
# df_filtered_image

In [ ]:
# df_meta_image = pd.merge(df_meta, df_filtered_image, on="asin", how="inner")
# df_meta_image.shape

In [ ]:
df_meta_image = df_meta.copy()
df_meta_image.shape

In [ ]:
df_meta_image

In [ ]:
df_meta_image.to_parquet(os.path.join(PATH, "step4_image_feature_extract", "df_meta_image.parquet"), index=False)

In [ ]:
df_has_image = df_meta_image[df_meta_image["image_path"].notnull()].copy()
df_does_not_have_image = df_meta_image[df_meta_image["image_path"].isnull()].copy()
print("Số lượng item có ảnh: ", df_has_image.shape)
print("Số lượng item không có ảnh: ", df_does_not_have_image.shape)

In [ ]:
def read_image_features(path, feature_size):
    if not os.path.exists(path):
        return
    with open(path, "rb") as f:
        while True:
            asin_bytes = f.read(10)
            if not asin_bytes:
                break
            try:
                asin = asin_bytes.decode("utf-8").strip()
                a = array.array("f")
                # Đọc đúng số lượng float bạn yêu cầu
                a.fromfile(f, feature_size)
                yield asin, a.tolist()
            except EOFError:
                break
            except Exception as e:
                print(f"Lỗi tại vị trí {f.tell()}: {e}")
                break

In [ ]:
# --- CẤU HÌNH ---
FEATURE_SIZE = 768
FILE_B_PATH = os.path.join(PATH, "step4_image_feature_extract", "image_feature.open_ai_clip.b")

In [ ]:
image_features = read_image_features(FILE_B_PATH, feature_size=FEATURE_SIZE)

In [ ]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(image_features)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

In [ ]:
first_item[1][:10]

In [ ]:
def process_and_save_image_features(df, file_b_path, output_dir, feature_size=4096):
    """
    Xử lý trích xuất feature từ file .b, map vào itemID và xử lý dữ liệu thiếu.

    Args:
        df (pd.DataFrame): DataFrame chứa cột 'asin' và 'itemID'.
        file_b_path (str): Đường dẫn đến file .b chứa features.
        output_dir (str): Thư mục lưu file .npy và log.
        feature_size (int): Kích thước vector feature (mặc định VGG16 là 4096).
    """
    num_items = len(df)
    output_npy = os.path.join(output_dir, "image_feat.npy")
    err_log = os.path.join(output_dir, "missed_img_itemIDs.csv")

    # 1. Map ASIN -> itemID (Đảm bảo itemID là kiểu int để làm index)
    map_asin_itemID = dict(zip(df["asin"], df.index)) # Hoặc df["itemID"] nếu itemID chạy từ 0 đến N-1

    # 2. Khởi tạo ma trận và biến hỗ trợ
    final_matrix = np.zeros((num_items, feature_size), dtype=np.float32)
    filled_indices = set()
    running_sum = np.zeros(feature_size, dtype=np.float64)

    print(f"🚀 Đang xử lý ảnh cho {num_items} sản phẩm...")

    # 3. Đọc và điền dữ liệu
    # Lưu ý: Hàm read_image_features cần được định nghĩa trước hoặc import vào
    try:
        for asin, feat_list in tqdm(read_image_features(file_b_path, feature_size),
                                     total=num_items, desc="Mapping features"):
            if asin in map_asin_itemID:
                target_idx = int(map_asin_itemID[asin])
                feat_array = np.array(feat_list, dtype=np.float32)

                final_matrix[target_idx] = feat_array
                filled_indices.add(target_idx)
                running_sum += feat_array
    except Exception as e:
        print(f"❌ Lỗi khi đọc file feature: {e}")
        return None

    # 4. Xử lý dữ liệu thiếu (Imputation bằng Average Vector)
    all_indices = set(range(num_items))
    missing_indices = sorted(list(all_indices - filled_indices))

    if len(filled_indices) > 0:
        avg_vector = (running_sum / len(filled_indices)).astype(np.float32)

        if missing_indices:
            print(f"⚠️ Cảnh báo: Thiếu {len(missing_indices)} ảnh (~{len(missing_indices)/num_items:.1%}).")
            print(f"👉 Đang lấp đầy bằng vector trung bình và lưu log tại: {err_log}")

            # Điền nhanh bằng vectorization
            final_matrix[missing_indices] = avg_vector

            # Lưu log các ID thiếu
            np.savetxt(err_log, missing_indices, delimiter=",", fmt="%d")

        # 5. Lưu kết quả
        np.save(output_npy, final_matrix)
        print(f"✅ Thành công! Ma trận: {final_matrix.shape}")
        return final_matrix

    else:
        print("❌ LỖI: Không có bất kỳ ASIN nào khớp giữa file .b và DataFrame!")
        return None

In [ ]:
result = process_and_save_image_features(df_meta_image, FILE_B_PATH, os.path.join(PATH, "step4_image_feature_extract"), FEATURE_SIZE)
result.shape

In [ ]:
!cp "/content/data/baby/2014/step4_image_feature_extract/image_feat.npy" "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/data/AmazonData/baby/2014/image_feat.open_ai_clip.npy"